# Credit Risk & Loan Default Prediction System
## Exploratory Data Analysis, DuckDB SQL Analytics & Imbalance Diagnostics

**Author:** Senior Machine Learning Engineer & Credit Risk Modeler  
**Dataset:** Home Credit Default Risk (`application_train.csv` & `bureau.csv`)  
**Objective:** Identify key risk drivers of payment difficulty (`TARGET = 1`), analyze target imbalance, engineer underwriting ratios, and perform DuckDB SQL segment analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

from src.data_loader import DataLoader
from src.feature_engineering import FeatureEngineer

print("Libraries loaded successfully.")

### 1. Data Ingestion & Shape Verification

In [ ]:
loader = DataLoader(data_dir="data/raw")
df_app = loader.load_application_train(nrows=50000)
df_bureau = loader.load_bureau(nrows=100000)

print(f"Application Dataset Shape: {df_app.shape[0]:,} rows, {df_app.shape[1]} columns")
print(f"Bureau Dataset Shape: {df_bureau.shape[0]:,} rows, {df_bureau.shape[1]} columns")
df_app.head(3)

### 2. Target Imbalance Analysis
In credit underwriting, defaulting borrowers are a minority class. In the Home Credit dataset, only **~8.07%** of applicants defaulted (`TARGET = 1`).

> **Why Accuracy is an Inadequate Metric:** A naive baseline model predicting `0` (Repaid) for every applicant achieves **91.93% accuracy**, yet catches **0% of defaulters**, incurring catastrophic loan charge-off losses. Underwriters must evaluate **ROC-AUC**, **PR-AUC (Precision-Recall Area Under Curve)**, and **Recall at an operationally viable Precision ceiling**.

In [ ]:
target_counts = df_app['TARGET'].value_counts()
default_rate = df_app['TARGET'].mean()

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(['Repaid (TARGET=0)', 'Defaulted (TARGET=1)'], target_counts.values, color=['#10B981', '#EF4444'], width=0.45)
ax.set_title(f"Target Distribution (Default Rate: {default_rate * 100:.2f}%)", fontsize=12, fontweight='bold')
ax.set_ylabel("Applicant Count")
for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + 500, f"{h:,} ({h/len(df_app)*100:.1f}%)", ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### 3. DuckDB SQL Analytical Queries
We register `application_train` in DuckDB and query segment default rates across contracts, education, and financial leverage.

In [ ]:
con = duckdb.connect(database=':memory:')
con.register('application_train', df_app)

# SQL Query: Default Rate by Education Type
edu_query = """
SELECT 
    NAME_EDUCATION_TYPE AS education_level,
    COUNT(*) AS total_applicants,
    SUM(TARGET) AS total_defaulters,
    ROUND(AVG(TARGET) * 100, 2) AS default_rate_pct,
    ROUND(AVG(AMT_INCOME_TOTAL), 2) AS avg_income
FROM application_train
GROUP BY NAME_EDUCATION_TYPE
ORDER BY default_rate_pct DESC;
"""
df_edu = con.execute(edu_query).df()
df_edu

**Business Interpretation:** Lower secondary education exhibits a **10.9%** default rate, whereas Higher education applicants experience only **5.4%** default rate. Higher educational attainment correlates with income stability and resilience to macro-economic shocks.

### 4. Feature Engineering: Domain Ratios

In [ ]:
fe = FeatureEngineer()
df_app_feat = fe.compute_application_features(df_app)

print("Engineered Features:")
df_app_feat[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'PAYMENT_RATE', 'AGE_YEARS', 'EMPLOYED_YEARS', 'EXT_SOURCES_MEAN']].head()

### 5. Correlation Heatmap for Core Numeric Features

In [ ]:
selected_numeric = [
    'TARGET', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'PAYMENT_RATE',
    'AGE_YEARS', 'EMPLOYED_YEARS', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]
corr_matrix = df_app_feat[selected_numeric].corr()

plt.figure(figsize=(9, 6.5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-0.4, vmax=0.4)
plt.title("Correlation Heatmap: Key Underwriting Indicators vs Default Target", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Key Correlation Findings:**
1. `EXT_SOURCE_1`, `EXT_SOURCE_2`, and `EXT_SOURCE_3` have the strongest negative correlation with default (-0.16 to -0.22), serving as the primary baseline risk signals.
2. `AGE_YEARS` has a negative correlation (-0.08); younger borrowers default at higher rates.
3. `CREDIT_INCOME_RATIO` and `ANNUITY_INCOME_RATIO` correlate positively with default risk, reflecting debt-service stress.